
#**In Silico Identification of Novel Compounds for Insomnia disease Step 1b: Enamine  GPCRs library ADME propierties molecule filtering**

##**Author: Maurizio Rafael Hernández Díaz. 2025-2026 Course**

Following the final part of the Notebook `"OX2_chembl_compounds_retrival_ADME_propierties.ipynb"` lipinksi rules are applied to the selected Enamine GPCRs library  that will be used in following steps as the library to apply the virtual screening of novel inhibitors for the Orexin 2 receptors(OX2) in order to obtain  as favorable molecules that present desireable physiochemical profiles for drug bioavailability .

##**0.-LIBRARIES REQUIRED:**

In [3]:
!pip install rdkit

In [4]:
#Importing required modules from the libraries:
import numpy as np
import matplotlib.pyplot as plt
import rdkit.Chem as chem
from typing import List
from io import StringIO
import pandas as pd
from rdkit.Chem import Descriptors as chemdesc
from rdkit.Chem import PandasTools
from rdkit.Chem import AllChem as achem
PandasTools.InstallPandasTools()
import json
import requests
import pandas as pd
import rdkit.Chem as chem
from rdkit.Chem import AllChem, Descriptors, Lipinski, MACCSkeys
from rdkit import Chem
from rdkit.Chem import QED

##**1.-MOLECULAR FILTERING: ADME PROPERTIES(Lipinski rules)**

As the Lipinksi rules states:  

– The molecule should have no more than 5 H-bond donors and 10 H-bond acceptors (i.e. N or O atoms).

– The molecular weight should be under 500 Daltons.(This parameter is really import as for the target it is needed to cross the blood-brain barrier) but I will appy a more restrictive threwshold of 400 due to the limitations in blood difusion due to such barrier.

– Its logP should be 5 maximum.

In [5]:
Enamine_dataset = pd.read_csv("../DATA/ENAMINE.csv", sep=",", skiprows=1)

In [6]:
Enamine_dataset
Enamine_dataset = Enamine_dataset.dropna(subset=["SMILES"])
Enamine_dataset

,SMILES,Catalog ID,MW,MW (desalted),ClogP,logS,HBD,HBA,TPSA,RotBonds,AnalogsFromREAL
0,O=C(NC1=NC(=NS1)C=2C=CC=CC2)C=3C(F)=CC=CC3Br,Z1343597788,378.220,378.220,3.017,-7.078,1,3,54.88,4,https://real.enamine.net/public-enum-files/Z13...
1,CC1CC=2C=CC=CC2N1C(=O)C=3C=CN(N3)C=4C=CC(F)=CC4,Z391927422,321.349,321.349,3.646,-4.672,0,2,38.13,3,https://real.enamine.net/public-enum-files/Z39...
2,CC=1C=CC(=CC1)C=2N=NN(CC(=O)NC=3C=CC=CC3OCC(F)...,Z241875964,391.348,391.348,3.408,-5.779,1,5,81.93,7,https://real.enamine.net/public-enum-files/Z24...
3,FC=1C=CC=C(C1)NCC2=NC(=NO2)C3=CC=CO3,Z297575978,259.236,259.236,2.370,-3.198,1,3,64.09,4,https://real.enamine.net/public-enum-files/Z29...
4,CC=1C=CC=CC1NC(=O)C2CCCN2C3=NC=CS3,Z391936202,287.382,287.382,2.052,-3.989,1,3,45.23,3,https://real.enamine.net/public-enum-files/Z39...
...,...,...,...,...,...,...,...,...,...,...,...
53435,CCN1C(CN2N=CSC2=O)=NC=3C(F)=CC=CC31,Z1822510276,278.307,278.307,2.708,-2.728,0,3,50.49,3,https://real.enamine.net/public-enum-files/Z18...
53436,COC=1C=C2C(=NC=NC2=CC1F)N[C@@H](C)C=3C=CC(Cl)=CN3,Z1980873798,332.760,332.760,3.301,-4.545,1,5,59.93,4,https://real.enamine.net/public-enum-files/Z19...
53437,CN(CC1=NC(=NO1)C=2C=CC=NC2)C=3C=CC=CC3OC=4C=CC...,Z1207932493,358.394,358.394,4.220,-4.670,0,4,64.28,6,https://real.enamine.net/public-enum-files/Z12...
53438,O=C(C=1C=CN(N1)C=2C=CC=CC2)N3CCCCC3CN4C=CC=N4,Z1139313763,335.404,335.404,2.372,-3.120,0,3,55.95,4,https://real.enamine.net/public-enum-files/Z11...


In [7]:
Enamine_dataset["mol"] = Enamine_dataset["SMILES"].apply(Chem.MolFromSmiles)

In [8]:
Enamine_dataset.columns

Index(['SMILES', 'Catalog ID', 'MW', 'MW (desalted)', 'ClogP', 'logS', 'HBD',
       'HBA', 'TPSA', 'RotBonds', 'AnalogsFromREAL', 'mol'],
      dtype='str')

In [9]:
def lipinski_filter(mol):
    if mol is None:
        return False


    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)

    conditions = [
        mw < 400,
        logp < 5.0,
        hbd < 5,
        hba < 10
    ]

    return all(conditions)


Enamine_dataset["is_lipinski"] = Enamine_dataset["mol"].apply(lipinski_filter)

print(Enamine_dataset["is_lipinski"].value_counts())

is_lipinski
True     48520
False     4920
Name: count, dtype: int64


In [10]:
Enamine_dataset=Enamine_dataset[Enamine_dataset["is_lipinski"]==True]
Enamine_dataset

,SMILES,Catalog ID,MW,MW (desalted),ClogP,logS,HBD,HBA,TPSA,RotBonds,AnalogsFromREAL,mol,is_lipinski
0,O=C(NC1=NC(=NS1)C=2C=CC=CC2)C=3C(F)=CC=CC3Br,Z1343597788,378.220,378.220,3.017,-7.078,1,3,54.88,4,https://real.enamine.net/public-enum-files/Z13...,<rdkit.Chem.rdchem.Mol object at 0x15fdd4820>,True
1,CC1CC=2C=CC=CC2N1C(=O)C=3C=CN(N3)C=4C=CC(F)=CC4,Z391927422,321.349,321.349,3.646,-4.672,0,2,38.13,3,https://real.enamine.net/public-enum-files/Z39...,<rdkit.Chem.rdchem.Mol object at 0x15fdd4890>,True
2,CC=1C=CC(=CC1)C=2N=NN(CC(=O)NC=3C=CC=CC3OCC(F)...,Z241875964,391.348,391.348,3.408,-5.779,1,5,81.93,7,https://real.enamine.net/public-enum-files/Z24...,<rdkit.Chem.rdchem.Mol object at 0x15fdd4900>,True
3,FC=1C=CC=C(C1)NCC2=NC(=NO2)C3=CC=CO3,Z297575978,259.236,259.236,2.370,-3.198,1,3,64.09,4,https://real.enamine.net/public-enum-files/Z29...,<rdkit.Chem.rdchem.Mol object at 0x15fdd4970>,True
4,CC=1C=CC=CC1NC(=O)C2CCCN2C3=NC=CS3,Z391936202,287.382,287.382,2.052,-3.989,1,3,45.23,3,https://real.enamine.net/public-enum-files/Z39...,<rdkit.Chem.rdchem.Mol object at 0x15fdd49e0>,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
53435,CCN1C(CN2N=CSC2=O)=NC=3C(F)=CC=CC31,Z1822510276,278.307,278.307,2.708,-2.728,0,3,50.49,3,https://real.enamine.net/public-enum-files/Z18...,<rdkit.Chem.rdchem.Mol object at 0x326afe8f0>,True
53436,COC=1C=C2C(=NC=NC2=CC1F)N[C@@H](C)C=3C=CC(Cl)=CN3,Z1980873798,332.760,332.760,3.301,-4.545,1,5,59.93,4,https://real.enamine.net/public-enum-files/Z19...,<rdkit.Chem.rdchem.Mol object at 0x326afe960>,True
53437,CN(CC1=NC(=NO1)C=2C=CC=NC2)C=3C=CC=CC3OC=4C=CC...,Z1207932493,358.394,358.394,4.220,-4.670,0,4,64.28,6,https://real.enamine.net/public-enum-files/Z12...,<rdkit.Chem.rdchem.Mol object at 0x326afe9d0>,True
53438,O=C(C=1C=CN(N1)C=2C=CC=CC2)N3CCCCC3CN4C=CC=N4,Z1139313763,335.404,335.404,2.372,-3.120,0,3,55.95,4,https://real.enamine.net/public-enum-files/Z11...,<rdkit.Chem.rdchem.Mol object at 0x326afea40>,True


**Initially I only consiered Lipinski rules in order to filter the Enamine compounds but seaqrching in the litertature I found that considering my target presents special characteristics as being part of the Central Nervous system the posible lead needs to present special characteristics as described by Pajouhesh  & Lenz, 2005 where the authors describe chemical propierties of succesful central nervous system drugs generating stricter criteria in comparassion to my initial filter.**

In [11]:
df=Enamine_dataset.copy()

In [12]:
def pajouhesh_lenz_filter(df):
    """
    Filter based on  Pajouhesh & Lenz (2005)
    for succesful CNS drugs.
    """
    Enamine_dataset["RotatableBonds"] = Enamine_dataset["mol"].apply(Descriptors.NumRotatableBonds)
    Enamine_dataset["mw"] = Enamine_dataset["mol"].apply(Descriptors.MolWt)
    Enamine_dataset["logp"] = Enamine_dataset["mol"].apply(Descriptors.MolLogP)
    Enamine_dataset["hbd"] = Enamine_dataset["mol"].apply(Descriptors.NumHDonors)
    Enamine_dataset["hba"] = Enamine_dataset["mol"].apply(Descriptors.NumHAcceptors)
    Enamine_dataset["TPSA"] = Enamine_dataset["mol"].apply(Descriptors.TPSA)
    df=Enamine_dataset.copy()
    cns_candidates = df[
        (df['mw'] < 400) &
        (df['logp'] >= 2.0) & (df['logp'] <= 4.0) &
        (df['TPSA'] <= 70) &
        (df['hbd'] <= 1) &
        (df['hba'] <= 7) &
        (df['RotatableBonds'] <= 7)
    ].copy()
    return cns_candidates

Enamine_SNC_ready = pajouhesh_lenz_filter(df)



In [13]:

df["QED"] = df["mol"].apply(QED.qed)
df= df[df["QED"] > 0.6].copy()

In [14]:
Enamine_SNC_ready=df.copy()

In [15]:
Enamine_SNC_ready.drop(columns=["is_lipinski"],inplace=True)

In [16]:
print(F"The number of molecules after  initial filter based on  Lipinski rules only  ",len(Enamine_dataset))
print(F"The number of molecules after Lipinski rules and Pajouhesh  & Lenz, 2005 and QED> 0.6 is ",len(Enamine_SNC_ready))

The number of molecules after  initial filter based on  Lipinski rules only   48520
The number of molecules after Lipinski rules and Pajouhesh  & Lenz, 2005 and QED> 0.6 is  45119


In [17]:
Enamine_SNC_ready.to_csv("../DATA/Enamine_curated_database.csv")